# 📱 Notebook 4: Phone Call Detection — YOLOv8 + CBAM
**Cairo University | FCAI | AI Department | 2025-2026**

## Task: Classify phone call gesture vs normal driving
- **Classes**: 2 (using_phone, not_using_phone)
- **Dataset**: Distracted Driver Dataset (Kaggle) — State Farm
- **Contribution**: YOLOv8-cls + CBAM

## Why this dataset?
The State Farm Distracted Driver dataset has:
- c0: safe driving
- c1: texting right
- c2: **talking on phone right** ← we use this
- c3: texting left
- c4: **talking on phone left** ← we use this
- etc.

We extract c2+c4 (phone call) vs c0 (safe driving) for binary classification

In [ ]:
!pip install ultralytics opendatasets -q
import torch; print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Download Dataset ───────────────────────────────────────────────────────
import opendatasets as od
# State Farm Distracted Driver Detection
od.download('https://www.kaggle.com/competitions/state-farm-distracted-driver-detection/data')
import os
# Check structure
base_dir = 'state-farm-distracted-driver-detection'
for d in os.listdir(base_dir):
    print(d)

In [ ]:
# ── Prepare Binary Dataset ─────────────────────────────────────────────────
# phone_call = c2 (phone right) + c4 (phone left)
# safe       = c0 (safe driving)
import os, glob, shutil, random
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

train_dir = 'state-farm-distracted-driver-detection/imgs/train'

# Phone call images (c2 + c4)
phone_imgs = (glob.glob(f'{train_dir}/c2/*.jpg') +
              glob.glob(f'{train_dir}/c4/*.jpg'))

# Safe driving (c0)
safe_imgs  = glob.glob(f'{train_dir}/c0/*.jpg')

print('📊 DATA BALANCE CHECK:')
print(f'   Phone call  : {len(phone_imgs)} images (c2+c4)')
print(f'   Safe driving: {len(safe_imgs)} images (c0)')

# Balance
min_count = min(len(phone_imgs), len(safe_imgs))
random.seed(42)
phone_imgs = random.sample(phone_imgs, min_count)
safe_imgs  = random.sample(safe_imgs,  min_count)
ratio = 1.0  # balanced after sampling
print(f'   ✅ Balanced to {min_count} per class')
print(f'   Total: {min_count*2} images')

plt.figure(figsize=(6, 4))
plt.bar(['Phone Call', 'Safe Driving'], [len(phone_imgs), len(safe_imgs)],
        color=['#F44336', '#4CAF50'])
plt.title('Dataset Balance — Phone Detection')
plt.ylabel('Image Count')
for i, v in enumerate([len(phone_imgs), len(safe_imgs)]):
    plt.text(i, v+5, str(v), ha='center', fontweight='bold')
plt.savefig('phone_balance.png', dpi=150)
plt.show()

# Create dataset structure
for split in ['train', 'val']:
    for cls in ['phone_call', 'safe_driving']:
        os.makedirs(f'phone_dataset/{split}/{cls}', exist_ok=True)

def split_copy(imgs, cls_name):
    train, val = train_test_split(imgs, test_size=0.2, random_state=42)
    for i, src in enumerate(train): shutil.copy(src, f'phone_dataset/train/{cls_name}/{cls_name}_{i}.jpg')
    for i, src in enumerate(val):   shutil.copy(src, f'phone_dataset/val/{cls_name}/{cls_name}_{i}.jpg')
    return len(train), len(val)

tr_p, v_p = split_copy(phone_imgs, 'phone_call')
tr_s, v_s = split_copy(safe_imgs,  'safe_driving')
print(f'✅ Split: Train={tr_p+tr_s} | Val={v_p+v_s}')

In [ ]:
# ── Visualize Samples ──────────────────────────────────────────────────────
import cv2, glob, random, matplotlib.pyplot as plt

phone_s = random.sample(glob.glob('phone_dataset/train/phone_call/*.jpg'), 3)
safe_s  = random.sample(glob.glob('phone_dataset/train/safe_driving/*.jpg'), 3)

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
fig.suptitle('Dataset Samples — Phone Detection', fontsize=13)

for ax, path in zip(axes[0], phone_s):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    ax.imshow(img); ax.set_title('Phone Call', color='red'); ax.axis('off')

for ax, path in zip(axes[1], safe_s):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    ax.imshow(img); ax.set_title('Safe Driving', color='green'); ax.axis('off')

plt.tight_layout()
plt.savefig('phone_samples.png', dpi=150)
plt.show()

In [ ]:
# ── CBAM ───────────────────────────────────────────────────────────────────
import torch, torch.nn as nn

class ChannelAttention(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        self.ap=nn.AdaptiveAvgPool2d(1); self.mp=nn.AdaptiveMaxPool2d(1)
        self.fc=nn.Sequential(nn.Conv2d(c,c//r,1,bias=False),nn.ReLU(),nn.Conv2d(c//r,c,1,bias=False))
        self.sg=nn.Sigmoid()
    def forward(self,x): return self.sg(self.fc(self.ap(x))+self.fc(self.mp(x)))

class SpatialAttention(nn.Module):
    def __init__(self,k=7):
        super().__init__()
        self.cv=nn.Conv2d(2,1,k,padding=k//2,bias=False); self.sg=nn.Sigmoid()
    def forward(self,x):
        return self.sg(self.cv(torch.cat([torch.mean(x,1,True),torch.max(x,1,True)[0]],1)))

class CBAM(nn.Module):
    def __init__(self,c,r=16,k=7):
        super().__init__()
        self.ca=ChannelAttention(c,r); self.sa=SpatialAttention(k)
    def forward(self,x): return x*self.sa(x*self.ca(x))

from ultralytics.nn import modules; modules.CBAM=CBAM
import ultralytics.nn.tasks as t; t.CBAM=CBAM
print('✅ CBAM ready!')

In [ ]:
# ── Train Baseline ─────────────────────────────────────────────────────────
from ultralytics import YOLO
m_base = YOLO('yolov8n-cls.pt')
r_base = m_base.train(
    data='phone_dataset', epochs=30, imgsz=224, batch=32,
    name='phone_baseline', project='runs/phone',
    device=0, optimizer='SGD', lr0=0.01, momentum=0.937,
    weight_decay=0.0005, warmup_epochs=3, freeze=8,
    patience=10, save=True, plots=True, verbose=False
)
base_acc = r_base.results_dict.get('metrics/accuracy_top1', 0)
print(f'✅ Baseline: {base_acc*100:.1f}%')

In [ ]:
# ── Train CBAM ─────────────────────────────────────────────────────────────
from ultralytics import YOLO
m_cbam = YOLO('yolov8n-cls.pt')
r_cbam = m_cbam.train(
    data='phone_dataset', epochs=30, imgsz=224, batch=32,
    name='phone_cbam', project='runs/phone',
    device=0, optimizer='SGD', lr0=0.008, momentum=0.937,
    weight_decay=0.0005, warmup_epochs=3, freeze=6,
    patience=10, save=True, dropout=0.2, verbose=False
)
cbam_acc = r_cbam.results_dict.get('metrics/accuracy_top1', 0)
print(f'✅ CBAM: {cbam_acc*100:.1f}%')
print(f'   Improvement: {(cbam_acc-base_acc)*100:+.2f}%')

In [ ]:
# ── Compare & Download ─────────────────────────────────────────────────────
import pandas as pd, matplotlib.pyplot as plt, shutil, os
from google.colab import files

def find_csv(kw):
    for root,d,fs in os.walk('runs/phone'):
        if kw in root:
            for f in fs:
                if f=='results.csv': return os.path.join(root,f)

df_b = pd.read_csv(find_csv('baseline')); df_b.columns = df_b.columns.str.strip()
df_c = pd.read_csv(find_csv('cbam'));     df_c.columns = df_c.columns.str.strip()
acc  = [c for c in df_b.columns if 'top1' in c.lower() and 'val' in c.lower()]

fig, ax = plt.subplots(figsize=(8, 5))
fig.suptitle('Phone Detection: YOLOv8 vs YOLOv8+CBAM\nCairo University 2026', fontsize=12)
if acc:
    ax.plot(df_b[acc[0]], label='Baseline', color='#2196F3', linewidth=2)
    ax.plot(df_c[acc[0]], label='CBAM', color='#FF5722', linewidth=2, linestyle='--')
    ax.legend(); ax.grid(True, alpha=0.3); ax.set_title('Val Accuracy')
plt.savefig('phone_comparison.png', dpi=150)
plt.show()

print('='*55)
print(f'  Dataset    : State Farm (phone_call vs safe)')
print(f'  Train      : {tr_p+tr_s} | Val: {v_p+v_s}')
print(f'  Baseline   : {base_acc*100:.1f}%')
print(f'  CBAM       : {cbam_acc*100:.1f}%')
print(f'  Improvement: {(cbam_acc-base_acc)*100:+.2f}%')
print('='*55)

for root,d,fs in os.walk('runs/phone/phone_cbam'):
    for f in fs:
        if f=='best.pt': shutil.copy(os.path.join(root,f),'phone_cbam.pt')

for f in ['phone_cbam.pt','phone_comparison.png','phone_balance.png','phone_samples.png']:
    if os.path.exists(f): files.download(f); print(f'✅ {f}')
print('🎉 Done! Put phone_cbam.pt in models/weights/')